# 04 — Paired Development Evaluation

Read the original measured policy–detector pair, or evaluate this experiment’s detector on its matching offline policy and the shared development IDs/GT.

**Inputs:** Committed historical matrix, or this experiment’s saved native checkpoint and prepared data.

**Outputs:** runs/v1/eval/metrics.json and native plots for explicit new inference; historical reads write nothing.

**Prerequisites:** Preparation and training are independent saved prerequisites. Evaluation never imports or invokes training.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy–Detector Pair](#section-2)
- [3. Evaluation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Historical evidence is read under its original attribution. New S20 inference needs explicit switches.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
from vehicle_detection.experiment_data import load_experiment

EXPERIMENT = "v1"
require_paths(config=ROOT / f"configs/{EXPERIMENT}.yaml", shared=ROOT / "configs/data.yaml")
config = load_experiment(ROOT, EXPERIMENT)
from vehicle_detection.evaluation import evaluate_experiment, historical_result

READ_HISTORICAL = True
ALLOW_INFERENCE = False
# Avoid the observed multiprocess loader stall in standalone evaluation.
EVAL_SETTINGS = {"workers": 0}
HISTORICAL_BRANCH = "adaptive"

<a id="section-2"></a>

## 2. Policy–Detector Pair

Both arms use workers=0 after the observed multiprocess loader stall; GPU inference settings are unchanged. Only development is evaluated. A checkpoint or data export from another experiment fails before inference.

In [ ]:
if READ_HISTORICAL:
    historical = historical_result(ROOT, HISTORICAL_BRANCH)
    result = historical["result"]
    print("Original measured evaluation:", historical["evaluation"])
    print("No new inference or training occurred.")
else:
    result = evaluate_experiment(
        ROOT, EXPERIMENT, allow_inference=ALLOW_INFERENCE, settings=EVAL_SETTINGS
    )
display(pd.DataFrame([result["overall"]]))
display(pd.DataFrame(result["classes"]))

<a id="section-3"></a>

## 3. Evaluation Summary

Native model timing excludes offline preprocessing/I/O. Single-seed historical development metrics do not establish a validated improvement.

In [ ]:
print("Native timings (ms):", result["speed_ms"])
print("Role:", result["role"], "Final images accessed:", result["final_test_accessed"])
print("Original reports: docs/development-five-v1/RESULTS.md and docs/ISSUE21_RESULTS.md")